In [ ]:
%load_ext watermark


In [ ]:
from IPython.display import display, HTML
from matplotlib import pyplot as plt
import pandas as pd
import polars as pl
import seaborn as sns
from slugify import slugify
from teeplot import teeplot as tp

from pylib._percentilestatcat_plot import (
    percentilestatcat_plot,
)
from pylib._seed_global_rngs import seed_global_rngs


In [ ]:
%watermark -diwmuv -iv


In [ ]:
teeplot_subdir = "2025-05-17-vanilla-compscreen"
teeplot_subdir


In [ ]:
seed_global_rngs(1)


## Get Data


In [ ]:
url = "https://osf.io/r8skg/download"
tmp_path = f"/tmp/{teeplot_subdir}.pqt"

print(f"Downloading data from {url}")

pl.scan_parquet(
    url,
    low_memory=True,
    retries=5,
).sink_parquet(tmp_path)
print("done!")


In [ ]:
df = pl.scan_parquet(
    tmp_path,
    low_memory=True,
    retries=5,
)
schema = df.collect_schema()


In [ ]:
fil = (
    df.filter(
        pl.col("trt_hsurf_bits").eq(0),
    )
    .filter(
        pl.col("replicate_uuid").eq(
            pl.col("replicate_uuid").first().over("trt_name"),
        )
    )
    .select(
        pl.exclude([k for k, v in schema.items() if v == pl.String]),
    )
    .collect()
)


## Plot Data


In [ ]:
data = fil.to_pandas()
cat = []
for y, y_name, comparator in (
    ["defmut_norm_all-num_leaves", "clade size", "norm all"],
    ["defmut_norm_ot_bin:month-num_leaves", "clade size", "norm monthly"],
    ["defmut_norm_all-clade_duration", "clade duration", "norm all"],
    [
        "defmut_norm_ot_bin:month-clade_duration",
        "clade duration",
        "norm monthly",
    ],
):
    catdf = data[data["is_focal_defmut"].astype(bool)]
    cat.append(
        pd.DataFrame(
            {
                "trt_name": catdf["trt_name"].str.replace("/", "\n").values,
                "metric": y_name,
                "metric_shift": {"clade size": 0.33, "clade duration": 1.33}[
                    y_name
                ],
                "comparator set": comparator,
                "percentile": catdf[y].values,
            },
        ),
    )

data = pd.concat(cat, ignore_index=True)


In [ ]:
with tp.teed(
    sns.catplot,
    data=data,
    y="percentile",
    col="trt_name",
    col_order=[
        "Sneu\nGneu",
        # "Sben1.1x\nGneu",
        # "Sben1.3x\nGneu",
        "Sben2x\nGneu",
        # "Sben1.1x\nGdel1.1x",
        # "Sben1.3x\nGdel1.3x",
        "Sben2x\nGdel2x",
    ],
    alpha=0.5,
    gap=0.3,
    hue="metric",
    row="comparator set",
    kind="boxen",
    x="metric",
    height=2.3,
    aspect=0.6,
    margin_titles=True,
    legend=True,
) as g:
    g.map_dataframe(
        sns.barplot,
        y="percentile",
        x="metric",
        alpha=0.0,
    )
    g.map_dataframe(
        sns.stripplot,
        y="percentile",
        x="metric_shift",
        hue="trt_name",
        hue_order=[
            "Sben2x\nGneu",
            "Sben2x\nGdel2x",
        ],
        palette=["black", "black"],
        s=1,
        alpha=0.1,
        native_scale=True,
    )
    g.map_dataframe(
        sns.stripplot,
        y="percentile",
        x="metric_shift",
        hue="trt_name",
        hue_order=[
            "Sneu\nGneu",
        ],
        palette=["black"],
        s=4,
        alpha=0.5,
        native_scale=True,
    )
    g.set_titles(col_template="{col_name}", row_template="{row_name}")
    g.set_axis_labels("", "percentile")
    g.fig.subplots_adjust(wspace=0.2, hspace=0.2)
    for ax in g.axes.flatten():
        ax.set_ylim(0, 100)
        ax.axhline(50, color="black", lw=1, ls="--")
        ax.set_xticks([])
        ax.set_xlabel("")

    # manually transcribed from below
    # https://github.com/mmore500/multilevel-selection-concept/blob/481a8a1da5c348d8fb27faafdf59aa4fb29817c2/binder/2025-05-17-vanilla-compscreen.ipynb
    pvals = [
        # norm all
        ["p=\n0.47", "p=\n0.48"],  # Sneu/Gneu
        ["p=\n0.004", "p=\n0.004"],  # Sben2x/Gneu
        ["p<\n0.001", "p<\n0.001"],  # Sben2x/Gdel2x
        # norm monthly
        ["p=\n0.59", "p=\n0.63"],  # Sneu/Gneu
        ["p=\n0.14", "p=\n0.13"],  # Sben2x/Gneu
        ["p<\n0.001", "p<\n0.001"],  # Sben2x/Gdel2x
    ]
    for ax, pv in zip(g.axes.flatten(), pvals):
        for x_pos, p_str in enumerate(pv):
            ax.text(
                x_pos,
                18,
                p_str,
                ha="center",
                va="top",
                fontsize=8,
                clip_on=False,
            )

    sns.move_legend(
        g,
        "lower center",
        bbox_to_anchor=(0.32, 0.03),
        ncol=2,
        title=None,
        frameon=False,
    )


In [ ]:
for (trt_name,), group in fil.group_by("trt_name"):
    display(HTML(f"<h1>{trt_name}</h1>"))

    dfx = group.to_pandas()
    dfx_ = group.to_pandas()
    dfx_["is_focal_defmut"] = "null"
    data = pd.concat([dfx, dfx_], ignore_index=True)

    for y in (
        "defmut_norm_all-num_leaves",
        # "defmut_norm_ot_bin:week-num_leaves",
        # "defmut_norm_ot_bin:fortnight-num_leaves",
        "defmut_norm_ot_bin:month-num_leaves",
        # "defmut_norm_ot_bin:quarter-num_leaves",
        # "defmut_norm_ot_bin:year-num_leaves",
        # "defmut_norm_match:variant_flavor-num_leaves",
        "defmut_norm_all-clade_duration",
        # "defmut_norm_ot_bin:week-clade_duration",
        # "defmut_norm_ot_bin:fortnight-clade_duration",
        "defmut_norm_ot_bin:month-clade_duration",
        # "defmut_norm_ot_bin:quarter-clade_duration",
        # "defmut_norm_ot_bin:year-clade_duration",
        # "defmut_norm_match:variant_flavor-clade_duration",
    ):
        display(HTML(f"<h2>{trt_name} {y}</h2>"))
        plt.close("all")  # release resources
        with tp.teed(
            percentilestatcat_plot,
            data=data,
            x="is_focal_defmut",
            y=y,
            hue="is_focal_defmut",
            teeplot_outattrs={
                "trt_name": slugify(trt_name),
            },
            teeplot_subdir=teeplot_subdir,
        ):
            pass


In [ ]:
for (trt_name,), group in fil.group_by("trt_name"):
    display(HTML(f"<h1>{trt_name}</h1>"))

    for norm in (
        "all",
        "match:variant_flavor",
        "ot_bin:week",
        "ot_bin:fortnight",
        "ot_bin:month",
        "ot_bin:quarter",
        "ot_bin:year",
    ):

        for var in "num_leaves", "clade_duration":
            display(HTML(f"<h2>{trt_name} {norm} {var}</h2>"))

            plt.close("all")  # release resources
            with tp.teed(
                sns.scatterplot,
                data=group,
                y=f"defmut_norm_{norm}-{var}",
                x=var,
                hue={
                    "all": None,
                    "ot_bin": "origin_time",
                    "match": "variant_flavor",
                }[norm.split(":")[0]],
                teeplot_outattrs={
                    "trt_name": slugify(trt_name),
                },
                teeplot_subdir=teeplot_subdir,
            ) as teed:
                teed.set_xscale(
                    {
                        "num_leaves": "log",
                        "clade_duration": "symlog",
                    }[var],
                )
                teed.set_ylim(-5, 105)
                teed.set_xlim(
                    {
                        "num_leaves": None,
                        "clade_duration": -0.5,
                    }[var],
                    None,
                )
                teed.spines[["right", "top"]].set_visible(False)
